In [45]:
# Utilities: loaders and validation
import os
import sys
import glob
import csv
from typing import List, Tuple
import numpy as np
import SimpleITK as sitk

def load_array(path: str) -> np.ndarray:
    ext = os.path.splitext(path)[1].lower()
    if ext != '.nrrd':
        raise ValueError(f"Unsupported file type for this notebook (expected .nrrd): {path}")
    image = sitk.ReadImage(path)
    arr = sitk.GetArrayFromImage(image).astype(np.float32)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D array, got shape {arr.shape} for {path}")
    return arr

def validate_same_shape(arrs: List[np.ndarray]) -> Tuple[int, int, int]:
    shapes = [a.shape for a in arrs]
    if len(set(shapes)) != 1:
        raise ValueError(f"All arrays must have the same shape, got: {shapes}")
    return arrs[0].shape

In [46]:
# Combine and export to CSV (skip all-zero rows) using pandas
import pandas as pd
import os
from pathlib import Path 

def nrrd2csv(input_files: List[str], output_csv: str):
  arrays = [load_array(p) for p in input_files]
  shape = validate_same_shape(arrays)
  N = len(arrays)

  # Stack as (N, X, Y, Z) then reshape to (voxels, N)
  stacked = np.stack(arrays, axis=0)  # (N, D, H, W)
  vox_mat = stacked.reshape(N, -1).T       # (D*H*W, N)

  # Filter: keep rows where at least one value != 0
  mask = ~(np.all(vox_mat == 0, axis=1))
  filtered = vox_mat[mask]

  # Create DataFrame and write CSV
  # Column names derived from final token before extension in filename
  # Example: original_firstorder_10Percentile.nrrd -> 10Percentile
  base_names = [os.path.splitext(os.path.basename(p))[0] for p in input_files]
  cols = [bn.split('_')[-1] if '_' in bn else bn for bn in base_names]
  df = pd.DataFrame(filtered, columns=cols)

  Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
  df.to_csv(output_csv, index=False)
  print(f"Saved CSV to: {output_csv} with columns: {cols}")

In [47]:
kernel = 1
className = 'firstorder'

In [ ]:
datasetPath = 'dataset/firstorder/kernel6-radius6/non_tumor'

patientFolders = os.listdir(datasetPath)
patientFolders.sort()
for patientFolder in patientFolders:
  if patientFolder.startswith('BraTS2021_'):
    patientPath = f"{datasetPath}/{patientFolder}"
    featureFiles = list(filter(lambda f: f.endswith('.nrrd'), os.listdir(patientPath)))
    featureFiles.sort()
    featureFiles = [f"{patientPath}/{featureFile}" for featureFile in featureFiles]
    outputCsvPath = f"{patientPath}/nrrd2csv.csv"
    if os.path.exists(outputCsvPath):
      print(f"CSV already exists for {patientFolder}, skipping.")
      for featureFile in featureFiles:
        os.remove(featureFile)
      continue
    nrrd2csv(featureFiles, outputCsvPath)
    for featureFile in featureFiles:
      os.remove(featureFile)

Saved CSV to: dataset/ngtdm/kernel6-radius6/non_tumor/BraTS2021_00002/nrrd2csv.csv with columns: ['Busyness', 'Coarseness', 'Complexity', 'Contrast', 'Strength']
